# Earnalism film_clips job `motion-pilot6-f01s014`

**Prepared, not run.** Input zip SHA256 `f3697f43e6243a2be4e1c85e3e89fbb536918312d840a14367234c0325e30264`. Set `RUN = True` in the first cell, then run all. With the R2 exchange (`exchange.json` in `/content/drive/MyDrive/earnalism-audiovisual/bicharak/motion-pilot6-f01s014/`) the notebook downloads `inputs.zip` itself and uploads the outputs when it finishes; without it, copy `inputs.zip` into that Drive folder first. Outputs and `manifest.json` (sha256 of every output) land in the same folder.

**Hard stop:** 6195 s (the reserved forecast) from the clock cell. No item starts after it and the runtime is unassigned at it; a stopped run's manifest says INCOMPLETE and cannot be imported.


In [ ]:
RUN = False

In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import pathlib
JOB_ID = 'motion-pilot6-f01s014'
TITLE = 'bicharak'
KIND = 'film_clips'
INPUT_ZIP_SHA256 = 'f3697f43e6243a2be4e1c85e3e89fbb536918312d840a14367234c0325e30264'
MODEL_REPO = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
MODEL_REVISION = 'resolved-at-run-time'
STEPS = 24
DRIVE_JOB_DIR = '/content/drive/MyDrive/earnalism-audiovisual/bicharak/motion-pilot6-f01s014'
# Hard wall-clock stop = the forecast seconds reserved on the ledger (R25)
DEADLINE_SECONDS = 6195
MANIFEST_RESERVE_S = 60
PACKAGES = ('diffusers', 'transformers', 'accelerate', 'safetensors', 'torch', 'huggingface_hub', 'imageio', 'ftfy')
EXCHANGE_SCHEMA = 'earnalism.colab-exchange.v1'
# The GPU cell stops before any download below this much GPU memory (0: any GPU)
MIN_VRAM_GIB = 21
GUIDANCE = 5.0
FRAMES = 121
FPS = 24
LICENCE_ALLOWLIST = ('apache-2.0',)
DEPTH_REPO = 'depth-anything/Depth-Anything-V2-Small-hf'
LICENCE_REPOS = (MODEL_REPO, DEPTH_REPO)
DIFFUSERS_PIN = 'diffusers==0.37.0'
# measured seconds per finished clip by resolution (this job's GPU class), for may_start
CLIP_S = {'hd': 900, 'sd': 435}


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import shutil, subprocess
# A GPU first: a CPU-only runtime would install, download and then fail with a confusing CUDA error.
_smi = shutil.which("nvidia-smi")
if not _smi or subprocess.run([_smi, "-L"], capture_output=True).returncode != 0:
    raise RuntimeError("No GPU attached: Runtime -> Disconnect and delete runtime, then Runtime -> "
                       "Change runtime type -> A100 GPU (or L4), Save, Connect, and Run all again")
print(subprocess.run([_smi, "-L"], capture_output=True, text=True).stdout.strip())
_min_gib = globals().get("MIN_VRAM_GIB", 0)
if _min_gib:  # the job's spec min_vram_gib: refuse a smaller GPU before any install or download
    _q = subprocess.run([_smi, "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                        capture_output=True, text=True)
    try:
        _gib = int(_q.stdout.split()[0]) / 1024  # MiB of the first GPU
    except (IndexError, ValueError):
        raise RuntimeError("could not read the GPU memory from nvidia-smi (" + repr(_q.stdout)
                           + "); this job needs at least %d GiB" % _min_gib) from None
    print("GPU memory %.1f GiB (this job needs at least %d GiB)" % (_gib, _min_gib))
    if _gib < _min_gib:
        raise RuntimeError("This job needs at least %d GiB of GPU memory and this GPU has %.1f GiB: it "
                           "needs an A100 or larger: Runtime -> Change runtime type -> A100, Save, "
                           "Connect, and Run all again (nothing was downloaded)" % (_min_gib, _gib))


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import os
_token, _secret_problem = None, None
try:
    from google.colab import userdata
    _token = userdata.get("HF_TOKEN") or None
except Exception as _e:  # no secret, notebook access not granted, or the prompt timed out
    _secret_problem = type(_e).__name__
    _token = os.environ.get("HF_TOKEN") or None
if _token:
    os.environ["HF_TOKEN"] = _token
from huggingface_hub import HfApi
try:
    HfApi().auth_check(MODEL_REPO, token=_token)
except Exception as _e:
    _fix = ("accept the model's terms at https://huggingface.co/" + MODEL_REPO + " with your Hugging "
            "Face account, put a Read token from https://huggingface.co/settings/tokens in the Colab "
            "secret HF_TOKEN (key icon in the left sidebar, Notebook access on), then Runtime > "
            "Restart session and Run all")
    if _secret_problem:
        _fix += " (the HF_TOKEN secret could not be read: %s; grant access if Colab asks)" % _secret_problem
    raise RuntimeError("No access to " + MODEL_REPO + " (" + type(_e).__name__ + "): " + _fix) from None
print("model access OK:", MODEL_REPO, "(token from HF_TOKEN)" if _token else "(public)")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import os
from huggingface_hub import HfApi
LICENCES = {}
for _repo in LICENCE_REPOS:
    try:
        HfApi().auth_check(_repo, token=os.environ.get("HF_TOKEN"))
        _info = HfApi().model_info(_repo, token=os.environ.get("HF_TOKEN"))
    except Exception as _e:
        raise RuntimeError("No access to " + _repo + " (" + type(_e).__name__ + "); nothing ran") from None
    _card = getattr(_info, "card_data", None) or getattr(_info, "cardData", None) or {}
    _lic = _card.get("license") if hasattr(_card, "get") else getattr(_card, "license", None)
    if not _lic:
        _lic = next((t.split(":", 1)[1] for t in (getattr(_info, "tags", None) or [])
                     if t.startswith("license:")), None)
    _lic = str(_lic).strip().lower() if _lic else None
    print("model card licence:", _repo, "->", _lic, "(allowlist: %s)" % ", ".join(LICENCE_ALLOWLIST))
    if _lic not in LICENCE_ALLOWLIST:
        raise RuntimeError("model card licence of " + _repo + " is " + repr(_lic) + ", not one of "
                           + ", ".join(LICENCE_ALLOWLIST) + ": this job never runs it; tell Claude")
    LICENCES[_repo] = _lic


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import importlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", DIFFUSERS_PIN, "transformers", "accelerate",
                "safetensors", "ftfy", "sentencepiece", "imageio", "imageio-ffmpeg"], check=True)
importlib.invalidate_caches()
import diffusers
print("diffusers", diffusers.__version__, "(pin: " + DIFFUSERS_PIN + ")")
if not hasattr(diffusers, "WanImageToVideoPipeline"):
    raise RuntimeError("diffusers " + diffusers.__version__ + " has no WanImageToVideoPipeline; "
                       "Runtime > Restart session, Run all")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import threading, time
if "JOB_STARTED" not in globals():
    JOB_STARTED = time.time()
    from google.colab import runtime
    STOP_GUARD = threading.Timer(DEADLINE_SECONDS, runtime.unassign)
    STOP_GUARD.daemon = True
    STOP_GUARD.start()
INCOMPLETE = False


def may_start(last_item_s):
    """Between items: start the next one only if it (and the manifest) fits before the hard stop."""
    global INCOMPLETE
    left = DEADLINE_SECONDS - (time.time() - JOB_STARTED)
    if left < last_item_s + MANIFEST_RESERVE_S:
        INCOMPLETE = True
        print("HARD STOP: %.0f s left of the %d s forecast; no further item starts" % (left, DEADLINE_SECONDS))
        return False
    return True


print("hard stop armed:", DEADLINE_SECONDS, "s from now")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, pathlib, zipfile
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = pathlib.Path(DRIVE_JOB_DIR)
INPUT_ZIP = OUT_DIR / "inputs.zip"
# R2 exchange (references/colab-exchange.md): exchange.json in this Drive folder carries two
# short-lived single-object URLs (GET inputs.zip, PUT outputs.tar). They are capabilities: never
# print them, never copy them elsewhere.
EXCHANGE = None
if (OUT_DIR / "exchange.json").is_file():
    import time, urllib.request
    from datetime import datetime
    EXCHANGE = json.loads((OUT_DIR / "exchange.json").read_text())
    if EXCHANGE.get("schema") != EXCHANGE_SCHEMA or EXCHANGE.get("job_id") != JOB_ID:
        raise RuntimeError("exchange.json belongs to a different job; ask Claude for this job's exchange.json")
    _left = datetime.fromisoformat(EXCHANGE["expires_at"].replace("Z", "+00:00")).timestamp() - time.time()
    if _left < DEADLINE_SECONDS + MANIFEST_RESERVE_S + 300:
        raise RuntimeError("the exchange URLs expire before this job's hard stop (%.0f s left); "
                           "ask Claude to re-sign them before running" % _left)
    if not INPUT_ZIP.is_file():
        with urllib.request.urlopen(EXCHANGE["inputs"]["url"], timeout=600) as _r:
            if not 200 <= _r.status < 300:
                raise RuntimeError("inputs.zip download failed: HTTP %d" % _r.status)
            _data = _r.read()
        _sha = hashlib.sha256(_data).hexdigest()
        if _sha != INPUT_ZIP_SHA256 or _sha != EXCHANGE["inputs"]["sha256"]:
            raise RuntimeError("downloaded inputs.zip sha256 mismatch: " + _sha + " (not kept)")
        INPUT_ZIP.write_bytes(_data)
        print("inputs.zip downloaded from the exchange:", len(_data), "bytes")
if not INPUT_ZIP.is_file():
    raise RuntimeError("Copy inputs.zip into " + str(OUT_DIR) + " first")
actual = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
if actual != INPUT_ZIP_SHA256:
    raise RuntimeError("inputs.zip sha256 mismatch: " + actual)
with zipfile.ZipFile(INPUT_ZIP) as z:
    JOB = json.loads(z.read("job.json"))
if JOB["job_id"] != JOB_ID:
    raise RuntimeError("job.json belongs to a different job")
print("inputs verified for", JOB_ID)


In [ ]:
# Film clips: Wan 2.2 TI2V-5B image-to-video (Apache-2.0), one ~5 s clip (121 frames, 24 fps) per item,
# plus Depth Anything V2 Small (Apache-2.0) depth maps for the camera glide: of every input still and of
# every clip's last frame. Per item: <key>.mp4 (near-lossless), <key>.last.png, <key>.last-depth.png.
# Per still: <image-stem>.depth.png (16-bit, 65535 = nearest). done.json in the Drive folder is the resume
# journal: a rerun skips every key whose mp4 sha256 is recorded there and still matches the file.
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
# Only the small Apache-2.0 depth model: Base and Large are non-commercial.
if DEPTH_REPO != "depth-anything/Depth-Anything-V2-Small-hf":
    raise RuntimeError("film_clips depth must use depth-anything/Depth-Anything-V2-Small-hf (Apache-2.0); "
                       "Base and Large are non-commercial, not " + repr(DEPTH_REPO))
import os, gc, time, zipfile, hashlib, json, pathlib
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


# --- journal helper begin (plain functions; the resume test execs this block alone) ---
def _file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _read_done(out_dir):
    try:
        done = json.loads((pathlib.Path(out_dir) / "done.json").read_text())
    except (OSError, ValueError):
        return {}
    return done if isinstance(done, dict) else {}


def remaining_keys(out_dir, keys):
    """The keys still to make: a key is finished only if done.json records the sha256 of its mp4 and
    the mp4 on Drive still has exactly that hash."""
    out_dir, done = pathlib.Path(out_dir), _read_done(out_dir)
    left = []
    for key in keys:
        mp4 = out_dir / (key + ".mp4")
        if not (mp4.is_file() and done.get(key) == _file_sha256(mp4)):
            left.append(key)
    return left


def record_done(out_dir, key):
    """Journal a finished clip (call it after every file of the key is written)."""
    out_dir = pathlib.Path(out_dir)
    done = _read_done(out_dir)
    done[key] = _file_sha256(out_dir / (key + ".mp4"))
    tmp = out_dir / "done.json.tmp"
    tmp.write_text(json.dumps(done, indent=1, sort_keys=True) + "\n")
    tmp.replace(out_dir / "done.json")
# --- journal helper end ---


import numpy as np
import torch
from PIL import Image

ITEMS = JOB["items"]
IMAGES = sorted({i["image"] for i in ITEMS if i.get("image")})
EXPECTED = ([k + suffix for k in (i["key"] for i in ITEMS) for suffix in (".mp4", ".last.png", ".last-depth.png")]
            + [n.rsplit(".", 1)[0] + ".depth.png" for n in IMAGES])
SECONDS = {}
gpu = torch.cuda.get_device_properties(0)
MODEL_INFO = {"repo": MODEL_REPO, "revision": None, "steps": STEPS, "guidance": GUIDANCE, "frames": FRAMES,
              "fps": FPS, "gpu": gpu.name, "depth_repo": DEPTH_REPO,
              "licences": {r: LICENCES.get(r) for r in LICENCE_REPOS}}

# Depth first (small): the stills' maps, then each clip's last frame as it is made.
from transformers import pipeline as _hf_pipeline
DEPTH = _hf_pipeline("depth-estimation", model=DEPTH_REPO, device=0)


def save_depth(img, name):
    """Normalised depth as a 16-bit PNG at the image's size (65535 = nearest)."""
    d = np.asarray(DEPTH(img)["predicted_depth"].squeeze().float().cpu(), dtype=np.float32)
    d = (d - d.min()) / max(1e-6, float(d.max() - d.min()))
    d = np.asarray(Image.fromarray(d).resize(img.size, Image.BICUBIC))
    Image.fromarray((np.clip(d, 0.0, 1.0) * 65535).astype(np.uint16)).save(OUT_DIR / name)


with zipfile.ZipFile(INPUT_ZIP) as z:
    for name in IMAGES:
        target = name.rsplit(".", 1)[0] + ".depth.png"
        if not (OUT_DIR / target).is_file():
            save_depth(Image.open(z.open(name)).convert("RGB"), target)
print("depth maps written for the stills:", len(IMAGES))

todo = remaining_keys(OUT_DIR, [i["key"] for i in ITEMS])
print("clips to make:", len(todo), "of", len(ITEMS), "(done.json skips the finished ones)")
if todo:
    from huggingface_hub import HfApi
    import diffusers
    from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
    from diffusers.utils import export_to_video
    revision = HfApi().model_info(MODEL_REPO).sha
    MODEL_INFO["revision"] = revision
    print("GPU", gpu.name, "%.1f GiB" % (gpu.total_memory / 2**30), "model", MODEL_REPO, revision[:12])
    vae = AutoencoderKLWan.from_pretrained(MODEL_REPO, revision=revision, subfolder="vae", torch_dtype=torch.float32)
    pipe = WanImageToVideoPipeline.from_pretrained(MODEL_REPO, revision=revision, vae=vae,
                                                   torch_dtype=torch.bfloat16, expand_timesteps=True)
    pipe.vae.enable_tiling()
    if gpu.total_memory >= 60 * 2**30:
        pipe.to("cuda")
    else:
        pipe.enable_model_cpu_offload()


# --- estimate helper begin (plain function; the may_start test execs this block alone) ---
def clip_need(item, last_by_res):
    """Seconds the next clip needs: the measured figure for its resolution, or the slowest clip of that
    resolution made so far in this run if that took longer."""
    res = "hd" if item["width"] >= 1280 else "sd"
    return max(float(CLIP_S[res]), last_by_res.get(res, 0.0))
# --- estimate helper end ---


def fit(img, w, h):
    """Centre-crop to the target aspect, then resize (Lanczos)."""
    sw, sh = img.size
    if sw / sh > w / h:
        nw = round(sh * w / h)
        img = img.crop(((sw - nw) // 2, 0, (sw - nw) // 2 + nw, sh))
    else:
        nh = round(sw * h / w)
        img = img.crop((0, (sh - nh) // 2, sw, (sh - nh) // 2 + nh))
    return img.resize((w, h), Image.LANCZOS)


def start_image(zf, item):
    """A continuation starts from the earlier clip's last frame; any other item from its still."""
    if item.get("continue_from"):
        last_png = OUT_DIR / (item["continue_from"] + ".last.png")
        if not last_png.is_file():
            raise RuntimeError(item["key"] + " continues " + item["continue_from"] + ", whose last frame "
                               "is not in the Drive folder; make that clip first")
        return Image.open(last_png).convert("RGB")
    return Image.open(zf.open(item["image"])).convert("RGB")


last_by_res = {}  # slowest clip so far per resolution ("hd"/"sd"), seconds
with zipfile.ZipFile(INPUT_ZIP) as z:
    for item in ITEMS:
        if item["key"] not in todo:
            continue
        if not may_start(clip_need(item, last_by_res)):
            break
        started = time.time()
        img = fit(start_image(z, item), item["width"], item["height"])
        frames = pipe(image=img, prompt=item["prompt"], negative_prompt=JOB["negative_prompt"],
                      height=item["height"], width=item["width"], num_frames=FRAMES,
                      num_inference_steps=STEPS, guidance_scale=GUIDANCE,
                      generator=torch.Generator("cpu").manual_seed(item["seed"])).frames[0]
        # near-lossless (quality=10): the default export is ~300 kbps and softens the picture, so there is
        # no fallback; a clip that cannot be saved near-lossless stops the job and is never journaled.
        try:
            export_to_video(frames, str(OUT_DIR / (item["key"] + ".mp4")), fps=FPS, quality=10)
        except TypeError as e:
            raise RuntimeError("export_to_video does not accept quality=10 (near-lossless) in diffusers "
                               + diffusers.__version__ + ": " + str(e) + "; nothing softened was written") from None
        _last = frames[-1]
        if not isinstance(_last, Image.Image):
            _arr = np.asarray(_last)
            _last = Image.fromarray((_arr * 255).astype(np.uint8) if _arr.dtype != np.uint8 else _arr)
        _last.save(OUT_DIR / (item["key"] + ".last.png"))
        save_depth(_last, item["key"] + ".last-depth.png")
        record_done(OUT_DIR, item["key"])  # last: every file of the key is on Drive
        took = time.time() - started
        _res = "hd" if item["width"] >= 1280 else "sd"
        last_by_res[_res] = max(last_by_res.get(_res, 0.0), took)
        SECONDS[item["key"]] = round(took, 1)
        print("wrote", item["key"], "%dx%d" % (item["width"], item["height"]), "in %.1f s" % took)
        gc.collect()
        torch.cuda.empty_cache()
MODEL_INFO["seconds_per_clip"] = SECONDS


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, time
import importlib.metadata as _md
outputs = {}
for p in sorted(OUT_DIR.iterdir()):
    if (p.is_file() and p.name not in ("manifest.json", "inputs.zip", "exchange.json", "done.json", "done.json.tmp")
            and p.suffix != ".ipynb"):  # Colab keeps the job notebook in this folder
        outputs[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
if not outputs:
    raise RuntimeError("no outputs were produced")
packages = {}  # R26: the resolved versions (and VCS commits) that actually ran
for _name in PACKAGES:
    try:
        _dist = _md.distribution(_name)
        _url = json.loads(_dist.read_text("direct_url.json") or "{}")
        _commit = _url.get("vcs_info", {}).get("commit_id")
        packages[_name] = _dist.version + (" @ " + _commit if _commit else "")
    except _md.PackageNotFoundError:
        packages[_name] = None
complete = not INCOMPLETE and all(name in outputs for name in EXPECTED)
manifest = {"schema": "earnalism.audiovisual-outputs.v1", "job_id": JOB_ID, "title": TITLE,
            "kind": KIND, "input_zip_sha256": INPUT_ZIP_SHA256, "model": MODEL_INFO,
            "status": "COMPLETE" if complete else "INCOMPLETE",
            "deadline_seconds": DEADLINE_SECONDS,
            "elapsed_seconds": round(time.time() - JOB_STARTED, 1),
            "packages": packages, "outputs": outputs}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print("manifest.json written:", manifest["status"], len(outputs), "files")
if not complete:
    print("INCOMPLETE: import refuses this folder; close the job with the CU Colab shows")
if globals().get("EXCHANGE") is not None:  # R2 exchange: one uncompressed tar, PUT once
    import os, tarfile, tempfile, urllib.error, urllib.request
    _fd, _tar = tempfile.mkstemp(suffix=".tar")  # local disk, never the Drive job folder
    os.close(_fd)
    with tarfile.open(_tar, "w", format=tarfile.PAX_FORMAT) as _t:
        for _name in ["manifest.json"] + sorted(outputs):
            _t.add(str(OUT_DIR / _name), arcname=_name, recursive=False)
    _size = os.path.getsize(_tar)
    _headers = dict(EXCHANGE["outputs"]["headers"], **{"Content-Length": str(_size)})
    with open(_tar, "rb") as _body:
        _req = urllib.request.Request(EXCHANGE["outputs"]["url"], data=_body, method="PUT",
                                      headers=_headers)
        try:
            with urllib.request.urlopen(_req, timeout=1800) as _r:
                _status = _r.status
        except urllib.error.HTTPError as _e:
            _status = _e.code
    os.unlink(_tar)
    print("outputs.tar upload: HTTP", _status, "(%d bytes)" % _size)
    if not 200 <= _status < 300:
        raise RuntimeError("outputs.tar upload failed: HTTP %d; the outputs stay in Drive, tell Claude"
                           % _status)
    print("outputs uploaded to the exchange; tell Claude the CU Colab shows")
